# Deep Past Challenge - Translate Akkadian to English

## Imports and config

In [12]:
import os
import re
import math
import time
import json
import random
import unicodedata
import platform

import numpy as np
import pandas as pd

import torch
import sacrebleu

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    total_mem_gb = gpu_props.total_memory / (1024**3)

    print("GPU name:", gpu_name)
    print("GPU compute capability:", f"{gpu_props.major}.{gpu_props.minor}")
    print("GPU total memory (GB):", f"{total_mem_gb:.2f}")
    print("CUDA version used by PyTorch:", torch.version.cuda)
    print("Number of CUDA devices:", torch.cuda.device_count())
else:
    print("Running on CPU")

CONFIG = {
    "train_path": "/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv",
    "test_path": "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv",
    "sample_sub_path": "/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv",

    "local_train_path": "train.csv",
    "local_test_path": "test.csv",
    "local_sample_sub_path": "sample_submission.csv",

    # preprocessing
    "lowercase": True,
    "normalize_unicode": True,
    "strip_extra_spaces": True,

    # pretrained model
    "model_name": "google/byt5-base",
    "prefix": "translate Akkadian to English: ",

    # tokenization / generation
    "max_source_length": 1000,
    "max_target_length": 1000,
    "max_new_tokens": 1000,
    "num_beams": 4,
    "length_penalty": 0.8,
    "no_repeat_ngram_size": 3,

    # training
    "batch_size": 8,
    "grad_accum": 2,
    "epochs": 20,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "patience": 4,

    # split
    "val_fraction": 0.15,

    # output
    "output_dir": "./akkadian_byt5",
    "submission_path": "submission.csv"
}

Device: cpu
Python version: 3.12.12
PyTorch version: 2.9.0+cpu
Running on CPU


## Data loading

In [13]:
def pick_existing_path(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"None of these paths exist: {paths}")

train_path = pick_existing_path(CONFIG["train_path"], CONFIG["local_train_path" ])
test_path = pick_existing_path(CONFIG["test_path"], CONFIG["local_test_path"])
sample_sub_path = pick_existing_path(CONFIG["sample_sub_path"], CONFIG["local_sample_sub_path"])

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

print(train_df.shape, test_df.shape, sample_sub.shape)
display(train_df.head())
display(test_df.head())
display(sample_sub.head())

(1561, 3) (4, 5) (4, 2)


,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,<gap> he did not give you a textile. He return...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...


,id,text_id,line_start,line_end,transliteration
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...
3,3,332fda50,25,30,me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...


,id,translation
0,0,"Thus Kanesh, say to the -payers, our messenge..."
1,1,In the letter of the City (it is written): Fro...
2,2,"As soon as you have heard our letter, who(ever..."
3,3,Send a copy of (this) letter of ours to every ...


## Text normalization

In [14]:
def normalize_source_text(text, normalize_unicode=True, strip_extra_spaces=True):
    text = "" if pd.isna(text) else str(text)

    if normalize_unicode:
        text = unicodedata.normalize("NFKC", text)

    text = text.replace("\n", " ").replace("\t", " ")

    # lowercase source only
    text = text.lower()

    # normalize common gap / missing-text markers
    text = text.replace("…", "<gap>")
    text = re.sub(r"\[\s*\.\.\.\s*\]", "<gap>", text)
    text = re.sub(r"\.\.\.+", "<gap>", text)
    text = re.sub(r"\[x+\]", "<gap>", text)
    text = re.sub(r"\bx\b", "<gap>", text)

    # normalize repeated gap markers
    text = re.sub(r"(<gap>\s*){2,}", "<gap> ", text)

    # normalize Unicode subscript digits to normal digits
    subscript_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    text = text.translate(subscript_map)

    # normalize Akkadian transliteration characters
    char_map = {
        "š": "sz",
        "ṣ": "s,",
        "ṭ": "t,",
        "ḫ": "h",
        "á": "a2",
        "à": "a3",
        "é": "e2",
        "è": "e3",
        "í": "i2",
        "ì": "i3",
        "ú": "u2",
        "ù": "u3",
    }
    for old, new in char_map.items():
        text = text.replace(old, new)

    if strip_extra_spaces:
        text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_target_text(text, normalize_unicode=True, strip_extra_spaces=True):
    text = "" if pd.isna(text) else str(text)

    if normalize_unicode:
        text = unicodedata.normalize("NFKC", text)

    text = text.replace("\n", " ").replace("\t", " ")

    if strip_extra_spaces:
        text = re.sub(r"\s+", " ", text).strip()

    return text


train_df["src"] = train_df["transliteration"].apply(
    lambda x: normalize_source_text(
        x,
        normalize_unicode=CONFIG["normalize_unicode"],
        strip_extra_spaces=CONFIG["strip_extra_spaces"],
    )
)

train_df["tgt"] = train_df["translation"].apply(
    lambda x: normalize_target_text(
        x,
        normalize_unicode=CONFIG["normalize_unicode"],
        strip_extra_spaces=CONFIG["strip_extra_spaces"],
    )
)

test_df["src"] = test_df["transliteration"].apply(
    lambda x: normalize_source_text(
        x,
        normalize_unicode=CONFIG["normalize_unicode"],
        strip_extra_spaces=CONFIG["strip_extra_spaces"],
    )
)

display(train_df[["transliteration", "translation", "src", "tgt"]].head())

,transliteration,translation,src,tgt
0,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...","kiszib ma-nu-ba-lu2m-a-szur dumu s,i2-la2-{d}i...","Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,1 tu2g sza qa2-tim i-tur4-dingir il5-qe2,Itūr-ilī has received one textile of ordinary ...
2,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,<gap> he did not give you a textile. He return...,tu2g u-la i-di2-na-ku-um i-tu3-ra-ma 9 gi2n ku...,<gap> he did not give you a textile. He return...
3,KIŠIB šu-{d}EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",kiszib szu-{d}en.li2l dumu szu-ku-bi-im kiszib...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,um-ma szu-ku-tum-ma a-na isztar-la2-ma-si2 u3 ...,From Šukkutum to Ištar-lamassī and Nitahšušar:...


## Train / validation split

In [15]:
idx = np.arange(len(train_df))
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

n_val = max(1, int(len(idx) * CONFIG["val_fraction"]))
val_idx = idx[:n_val]
tr_idx = idx[n_val:]

df_train = train_df.iloc[tr_idx].reset_index(drop=True)
df_val = train_df.iloc[val_idx].reset_index(drop=True)

print(df_train.shape, df_val.shape)

(1327, 5) (234, 5)


## Hugging Face dataset conversion

In [16]:
train_dataset = Dataset.from_pandas(df_train[["src", "tgt"]])
val_dataset = Dataset.from_pandas(df_val[["src", "tgt"]])
test_dataset = Dataset.from_pandas(test_df[["id", "src"]])

## Tokenizer and preprocessing

In [17]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def preprocess_train(examples):
    inputs = [CONFIG["prefix"] + str(x) for x in examples["src"]]
    targets = [str(x) for x in examples["tgt"]]

    model_inputs = tokenizer(
        inputs,
        max_length=CONFIG["max_source_length"],
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=CONFIG["max_target_length"],
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_test(examples):
    inputs = [CONFIG["prefix"] + str(x) for x in examples["src"]]
    model_inputs = tokenizer(
        inputs,
        max_length=CONFIG["max_source_length"],
        truncation=True,
    )
    return model_inputs

train_dataset_tok = train_dataset.map(preprocess_train, batched=True)
val_dataset_tok = val_dataset.map(preprocess_train, batched=True)
test_dataset_tok = test_dataset.map(preprocess_test, batched=True)

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1327 [00:00<?, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [18]:
def tokenizer_length_stats(texts, name):
    lengths = [len(tokenizer(CONFIG["prefix"] + str(x))["input_ids"]) for x in texts]
    s = pd.Series(lengths)
    print(f"{name} tokenized length stats:")
    print(s.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

tokenizer_length_stats(df_train["src"], "Train source")
tokenizer_length_stats(df_val["src"], "Val source")
tokenizer_length_stats(df_train["tgt"], "Train target")

Train source tokenized length stats:
count    1327.000000
mean      506.539563
std       295.148951
min        55.000000
50%       439.000000
75%       723.000000
90%      1013.400000
95%      1029.000000
99%      1042.000000
max      1065.000000
dtype: float64
Val source tokenized length stats:
count     234.000000
mean      517.025641
std       332.182611
min        60.000000
50%       448.000000
75%       811.500000
90%      1025.700000
95%      1035.000000
99%      1056.350000
max      1071.000000
dtype: float64
Train target tokenized length stats:
count    1327.000000
mean      535.116051
std       454.590569
min        38.000000
50%       422.000000
75%       709.500000
90%      1084.600000
95%      1383.100000
99%      2098.520000
max      4093.000000
dtype: float64


In [19]:
sample = train_dataset_tok[0]
print(sample.keys())
print(sample["input_ids"][:20])
print(sample["labels"][:20])

dict_keys(['src', 'tgt', 'input_ids', 'attention_mask', 'labels'])
[119, 117, 100, 113, 118, 111, 100, 119, 104, 35, 68, 110, 110, 100, 103, 108, 100, 113, 35, 119]
[73, 117, 114, 112, 35, 72, 113, 113, 100, 112, 48, 68, 200, 164, 200, 164, 120, 117, 35, 119]


## Model, collator, and metrics

In [20]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    CONFIG["model_name"],
    use_safetensors=True
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

model.generation_config.update(
    num_beams=CONFIG["num_beams"],
    no_repeat_ngram_size=CONFIG["no_repeat_ngram_size"],
    length_penalty=CONFIG["length_penalty"],
    early_stopping=True,
    max_new_tokens=CONFIG["max_new_tokens"],
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score
    chrf = sacrebleu.corpus_chrf(decoded_preds, [decoded_labels], word_order=2).score
    geo_mean = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0

    return {
        "bleu": bleu,
        "chrf++": chrf,
        "geo_mean": geo_mean,
    }

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Training arguments and trainer

In [21]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    learning_rate=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=1,
    predict_with_generate=False,
    load_best_model_at_end=True,
    greater_is_better=False,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=False,
    lr_scheduler_type="linear",
    warmup_steps=166,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tok,
    eval_dataset=val_dataset_tok,
    data_collator=data_collator,
)

In [ ]:
train_result = trainer.train()
print(train_result)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
best_ckpt = trainer.state.best_model_checkpoint
best_step = int(best_ckpt.split("-")[-1]) if best_ckpt is not None else None

best_epoch = None
for log in trainer.state.log_history:
    if "eval_loss" in log and "step" in log and log["step"] == best_step:
        best_epoch = log.get("epoch", None)
        break

print("Best checkpoint:", best_ckpt)
print("Best step:", best_step)
print("Best epoch:", best_epoch)
print("Best eval_loss:", trainer.state.best_metric)

## Compare greedy vs beam search on validation

In [ ]:
trainer.args.predict_with_generate = True

greedy_start = time.perf_counter()

pred_greedy = trainer.predict(
    val_dataset_tok,
    max_new_tokens=CONFIG["max_new_tokens"],
    num_beams=1,
)

greedy_end = time.perf_counter()
greedy_time = greedy_end - greedy_start

beam_start = time.perf_counter()

pred_beam = trainer.predict(
    val_dataset_tok,
    max_new_tokens=CONFIG["max_new_tokens"],
    num_beams=CONFIG["num_beams"],
)

beam_end = time.perf_counter()
beam_time = beam_end - beam_start

preds_greedy = tokenizer.batch_decode(pred_greedy.predictions, skip_special_tokens=True)
preds_beam = tokenizer.batch_decode(pred_beam.predictions, skip_special_tokens=True)
refs = df_val["tgt"].tolist()

metrics_greedy = {
    "bleu": sacrebleu.corpus_bleu(preds_greedy, [refs]).score,
    "chrf++": sacrebleu.corpus_chrf(preds_greedy, [refs], word_order=2).score,
}
metrics_greedy["geo_mean"] = math.sqrt(metrics_greedy["bleu"] * metrics_greedy["chrf++"]) if metrics_greedy["bleu"] > 0 and metrics_greedy["chrf++"] > 0 else 0.0

metrics_beam = {
    "bleu": sacrebleu.corpus_bleu(preds_beam, [refs]).score,
    "chrf++": sacrebleu.corpus_chrf(preds_beam, [refs], word_order=2).score,
}
metrics_beam["geo_mean"] = math.sqrt(metrics_beam["bleu"] * metrics_beam["chrf++"]) if metrics_beam["bleu"] > 0 and metrics_beam["chrf++"] > 0 else 0.0

print(f"Greedy time: {greedy_time:.2f}s ({greedy_time/60:.2f} min)")
print("Greedy:", metrics_greedy)

print(f"Beam time: {beam_time:.2f}s ({beam_time/60:.2f} min)")
print("Beam:", metrics_beam)

## Error analysis table

In [ ]:
analysis_df = pd.DataFrame({
    "src": df_val["src"].tolist(),
    "reference": refs,
    "pred_greedy": preds_greedy,
    "pred_beam": preds_beam,
})

analysis_df["src_len"] = analysis_df["src"].map(len)
analysis_df["ref_len"] = analysis_df["reference"].map(len)
analysis_df["greedy_exact"] = (analysis_df["reference"] == analysis_df["pred_greedy"])
analysis_df["beam_exact"] = (analysis_df["reference"] == analysis_df["pred_beam"])

display(analysis_df.sample(min(20, len(analysis_df)), random_state=SEED))

## Predict test set

In [ ]:
model.eval()

test_preds = []
batch_size = CONFIG["batch_size"]

test_inputs = [CONFIG["prefix"] + str(x) for x in test_df["src"].tolist()]

start_time = time.perf_counter()

for i in range(0, len(test_inputs), batch_size):
    batch_texts = test_inputs[i:i + batch_size]

    enc = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_source_length"],
    )

    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.no_grad():
        generated = model.generate(
            **enc,
            max_new_tokens=CONFIG["max_new_tokens"],
            num_beams=1,   # use greedy since it worked better for you
            length_penalty=CONFIG["length_penalty"],
            no_repeat_ngram_size=CONFIG["no_repeat_ngram_size"],
            early_stopping=True,
        )

    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
    test_preds.extend(decoded)

end_time = time.perf_counter()
print(f"Test prediction time: {end_time - start_time:.2f}s ({(end_time - start_time)/60:.2f} min)")

submission = pd.DataFrame({
    "id": test_df["id"].tolist(),
    "translation": test_preds,
})

display(submission.head())

## Save submission

In [ ]:
submission_path = CONFIG["submission_path"]
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()